# Temperature Analysis

This notebook uses the coverage-filtered annual, monthly, and seasonal datasets to analyze temperature patterns over time.

The 70% overall coverage threshold was applied in the previous notebook. Minimum temperature coverage (`tcoverage`) is retained as an additional indicator of data completeness but is not used to filter the analysis further.

The analysis focuses on mean temperature (`mean_temp`), minimum temperature (`mean_tmin`), and maximum temperature (`mean_tmax`).

## Imports and loading data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import linregress

processed_dir = Path("../data/processed")

annual = pd.read_csv(processed_dir / "annual_coverage.csv", index_col="year")

monthly = pd.read_csv(processed_dir / "monthly_coverage.csv",
                      index_col=["year", "month"])

seasonal = pd.read_csv(processed_dir / "seasonal_coverage.csv",
                       index_col=["season_year", "season"])

df = pd.read_csv(processed_dir / "sofia_daily_temperature.csv",
                 parse_dates=["time"])


## Annual Temperature Analysis

The analysis examines annual temperature over time, the long-term trend, and the warmest and coldest years.

In [ ]:
annual_means = (df.groupby("year").agg(mean_temp=("temp", "mean"),
                                       mean_tmin=("tmin", "mean"),
                                       mean_tmax=("tmax", "mean")))

annual_analysis = annual.join(annual_means)

### Average Temperature by Year

In [ ]:
annual_analysis[["mean_temp"]].plot(figsize=(12, 5), marker="o")

plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.title("Annual Mean Temperature Over Time")

plt.tight_layout()
plt.show()

### Annual Temperature Trends

Linear regression is used to estimate the long-term temperature trend over the study period.

The trend is expressed in °C per decade. A positive value indicates an increasing temperature trend, while a negative value indicates a decreasing trend.

Minimum temperature trends are calculated only for years with at least 70% Tmin coverage.

In [ ]:
annual_trends = {}

for column in ["mean_temp", "mean_tmax"]:

    data = annual_analysis.dropna(subset=[column])

    result = linregress(data.index, data[column])

    annual_trends[column] = {
        "trend_c_per_decade": result.slope * 10,
        "r_squared": result.rvalue**2
    }

# Tmin: only use periods with at least 70% Tmin coverage
tmin_data = annual_analysis[annual_analysis["tcoverage"] >= 70].dropna(
    subset=["mean_tmin"])

result = linregress(tmin_data.index, tmin_data["mean_tmin"])

annual_trends["mean_tmin"] = {
    "trend_c_per_decade": result.slope * 10,
    "r_squared": result.rvalue**2
}

annual_trends = pd.DataFrame(annual_trends).T

display(annual_trends)

Annual maximum temperatures show the strongest warming trend, increasing by approximately 0.43°C per decade. Mean temperature also shows a positive trend of approximately 0.26°C per decade, while minimum temperatures show a much weaker increase of approximately 0.09°C per decade. The higher R² for maximum and mean temperatures indicates that the linear trend explains considerably more of their year-to-year variation than it does for minimum temperatures.

### Warmest and Coldest Years

In [ ]:
print("Warmest years:")
display(
    annual_analysis.nlargest(10, "mean_temp")[[
        "coverage_percentage", "tcoverage", "mean_temp", "mean_tmin",
        "mean_tmax"
    ]])

print("Coldest years:")
display(
    annual_analysis.nsmallest(10, "mean_temp")[[
        "coverage_percentage", "tcoverage", "mean_temp", "mean_tmin",
        "mean_tmax"
    ]])

The warmest years in the dataset are concentrated in the period after 2000, with several of the highest annual mean temperatures occurring during the most recent decade. In contrast, the coldest years are generally concentrated before 2000, with a notable number occurring before 1980.

## Monthly Temperature Analysis

The analysis is performed separately for each calendar month to examine how temperature has changed over the years.

In [ ]:
monthly_means = (df.groupby(["year", "month"]).agg(mean_temp=("temp", "mean"),
                                                   mean_tmin=("tmin", "mean"),
                                                   mean_tmax=("tmax", "mean")))

monthly_analysis = monthly.join(monthly_means)

### Monthly Temperatures Across the Years


In [ ]:
months = [(1, "January"), (2, "February"), (3, "March"), (4, "April"),
          (5, "May"), (6, "June"), (7, "July"),
          (8, "August"), (9, "September"), (10, "October"), (11, "November"),
          (12, "December")]

for month_number, month_name in months:

    data = monthly_analysis[monthly_analysis.index.get_level_values("month") ==
                            month_number]

    data["mean_temp"].plot(figsize=(12, 5))

    plt.xlabel("Year")
    plt.ylabel("Mean Temperature (°C)")
    plt.title(f"{month_name} Mean Temperature Over Time")

    plt.tight_layout()
    plt.show()

### Monthly Temperature Trends

In [ ]:
monthly_trends = []
monthly_r_squared = []

for month_number, month_name in months:

    data = monthly_analysis[monthly_analysis.index.get_level_values("month") ==
                            month_number]

    # Mean temperature trend
    temp_data = data.dropna(subset=["mean_temp"])

    temp_result = linregress(temp_data.index.get_level_values("year"),
                             temp_data["mean_temp"])

    # Maximum temperature trend
    tmax_data = data.dropna(subset=["mean_tmax"])

    tmax_result = linregress(tmax_data.index.get_level_values("year"),
                             tmax_data["mean_tmax"])

    # Minimum temperature trend: tcoverage >= 70%
    tmin_data = data[data["tcoverage"] >= 70].dropna(subset=["mean_tmin"])

    tmin_result = linregress(tmin_data.index.get_level_values("year"),
                             tmin_data["mean_tmin"])

    monthly_trends.append({
        "month": month_name,
        "temp_trend_c_per_decade": temp_result.slope * 10,
        "tmin_trend_c_per_decade": tmin_result.slope * 10,
        "tmax_trend_c_per_decade": tmax_result.slope * 10
    })

    monthly_r_squared.append({
        "month": month_name,
        "temp_r_squared": temp_result.rvalue**2,
        "tmin_r_squared": tmin_result.rvalue**2,
        "tmax_r_squared": tmax_result.rvalue**2
    })

monthly_trends = pd.DataFrame(monthly_trends)
monthly_r_squared = pd.DataFrame(monthly_r_squared)
display(monthly_trends)
display(monthly_r_squared)

The monthly analysis shows a clear warming trend throughout the year, although the magnitude and consistency of warming vary between months. **Summer shows the strongest and most consistent warming**, with mean temperatures increasing by approximately **+0.45°C per decade in June, +0.52°C in July, and +0.58°C in August**. These months also have relatively high R² values, indicating that the positive trends are comparatively consistent over time. **Spring also shows substantial warming**, particularly in March, where mean temperature increases by about **+0.44°C per decade**. **Autumn shows more moderate warming**, with September increasing by **+0.27°C per decade** and October and November showing smaller trends. **Winter also exhibits clear warming**, especially in January (**+0.32°C per decade**) and February (**+0.27°C per decade**), although the lower R² values indicate greater year-to-year variability. Overall, the results suggest that **warming is occurring across all seasons, with the strongest and most consistent changes concentrated in the summer months, while winter and autumn warming is more variable**. Minimum and maximum temperatures generally follow the same warming pattern, although the magnitude differs between months.

## Seasonal Temperature Analysis

In [ ]:
df["season_year"] = df["year"]

df.loc[df["month"] == 12, "season_year"] += 1

seasonal_means = (df.groupby(["season_year",
                              "season"]).agg(mean_temp=("temp", "mean"),
                                             mean_tmin=("tmin", "mean"),
                                             mean_tmax=("tmax", "mean")))

seasonal_analysis = seasonal.join(seasonal_means)

### Seasonal Temperatures Across the Years


In [ ]:
seasons = ["Winter", "Spring", "Summer", "Autumn"]

for season in seasons:

    data = seasonal_analysis[
        seasonal_analysis.index.get_level_values("season") == season
    ]

    years = data.index.get_level_values("season_year")

    plt.figure(figsize=(12, 5))

    plt.plot(years, data["mean_temp"], label="Mean Temperature")
    plt.plot(years, data["mean_tmin"], label="Mean Minimum Temperature")
    plt.plot(years, data["mean_tmax"], label="Mean Maximum Temperature")

    plt.xlabel("Year")
    plt.ylabel("Temperature (°C)")
    plt.title(f"{season} Temperature Over Time")
    plt.legend()
    plt.tight_layout()
    plt.show()